In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin_min,silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler # each columns value get range 

In [ ]:
customers = pd.read_csv("retail_customers - retail_customers.csv")
print("Roaws,coloumns",customers.shape)
print("missing values",int(customers.isna().sum().sum()))
print()
customers.head()

In [ ]:
FEATURES = [
    "visits_last_year",
    "avg_order_value",
    "days_since_purchase",
    "online_order_share",
    "promo_emails_opened"
]
x = customers[FEATURES].copy()
print(x.std().round(3)) # we use this because we measure each column standard deviation high each column then we have to use standardscaler

In [ ]:
x_train,x_holdout = train_test_split(x,test_size=0.25,random_state=42)
print("Train raws:",len(x_train))
print("Held-out raws:",len(x_holdout))

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_holdout_scaled = scaler.transform(x_holdout)

print(x_train_scaled.std().round(3))


In [ ]:
k_values = range(2,9)
rows = []

for k in k_values:
    model = KMeans(n_clusters=k,init="k-means++",n_init=10,random_state=42)
    model.fit(x_train_scaled)
    
    train_mean_wcss = model.inertia_/len(x_train_scaled) # model.inertia = wcss
    print(f"{k}: train_mean_wcss: {train_mean_wcss}") # we have to get mean value of wcss because we have to get difference length test and holdout
    _,holdout_dist = pairwise_distances_argmin_min(x_holdout_scaled,model.cluster_centers_) # we find the text set wcss because no methods to find it we get each getted centroid distances and we get wcss
    holdout_mean_wcss = float((holdout_dist**2).mean())
    print(f"{k}:holdout_mean_wcss: {holdout_mean_wcss}")
    
    silhouette = silhouette_score(x_train_scaled,model.labels_) # measuer the cluster separations 1 is vey good and - values are bad
    print(f"{k}:silhoutte: {silhouette}")
    print("-"*50)
    
    rows.append({
        "k":k,
        "train_mean_wcss":train_mean_wcss,
        "holdout_mean_wcss":holdout_mean_wcss,
        "silhouette":silhouette,
        "iterations":model.n_iter_
    })
    
print(rows)
k_report = pd.DataFrame(rows)
k_report.round(3)



In [ ]:
model.cluster_centers_

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10 ,4))
axes[0].plot(k_report["k"], k_report["train_mean_wcss"], marker="o", label="Train")
axes[0].plot(k_report["k"], k_report["holdout_mean_wcss"], marker="o", label="Holdout")
axes[0].axvline(4, color="green", linestyle=":", label="Chosen k = 4")
axes[0].set_xlabel("number of clusters (k)")
axes[0].set_ylabel("mean WCSS")
axes[0].set_title("Train vs held-out tighness")
axes[0].set_xticks(list(k_values))
axes[0].legend()

axes[1].plot(k_report["k"], k_report["silhouette"], "o", color="tab:purple")
axes[1].axvline(4, color="green", linestyle=":")
axes[1].set_xlabel("number of clusters (k)")
axes[1].set_ylabel("silhouette score")
axes[1].set_title("Separation helper")
axes[1].set_xticks(list(k_values))

fig.tight_layout()
plt.show()

In [ ]:
CHOSEN_K = 4
production_scaler = StandardScaler()
x_scaled = production_scaler.fit_transform(x)

kmeans = KMeans(
    n_clusters=CHOSEN_K,
    init = "k-means++",
    n_init=10,
    random_state=42
)
kmeans.fit(x_scaled)
print("Iterations to converge :",kmeans.n_iter_)
print("Inertia (WCSS) on all customers:",round(kmeans.inertia_,1))
print("mean WCSS:",round(kmeans.inertia_/len(x_scaled),3))

In [ ]:
new_customer = pd.DataFrame([{
     "visits_last_year": 3,
    "avg_order_value": 310.0,
    "days_since_purchase": 80,
    "online_order_share": 0.30,
    "promo_emails_opened": 2
}])

In [ ]:
new_scaled = production_scaler.transform(new_customer)
new_cluster = kmeans.predict(new_scaled)[0]
print("Cluster id:",new_cluster)